In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta


### Settings

In [3]:
np.random.seed(42)
random.seed(42)

num_sessions = 350
start_date = datetime(2026, 7, 1)
end_date = datetime(2026, 7, 8)

user_roles = ["Customer", "Auditor", "Manufacturer", "Recycler", "Internal User"]
device_types = ["Desktop", "Mobile", "Tablet"]
browsers = ["Chrome", "Safari", "Edge", "Firefox"]
countries = ["Germany", "Netherlands", "France", "Italy", "Egypt"]

product_names = ["The White", "Steel Coil A", "Hot Rolled Coil", "Wire Rod", "Stainless Steel Sheet"]

tabs = [
    "General Information",
    "Paint Material",
    "Packaging",
    "Technical Documents",
    "Energy Overview",
    "Machine Energy",
    "Anomalies"
]

documents = [
    "Sustainability Data Sheet",
    "Technical Information",
    "Safety Data Sheet",
    "Interoperability Test Case Documentation",
    "Certificate of Conformity",
    "Energy Report"
]

exports = ["Export as PDF", "Export as JSON", "Export as CSV"]

verification_actions = ["Verify with Spherity"]

error_types = ["none", "broken_link", "failed_export", "failed_verification", "missing_data", "api_timeout"]

### Helper functions

In [4]:
def random_datetime(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)


def weighted_choice(options, weights):
    return random.choices(options, weights=weights, k=1)[0]

### Generate sessions table

In [5]:
sessions = []

for i in range(1, num_sessions + 1):
    session_id = f"S{i:04d}"
    user_role = weighted_choice(
        user_roles,
        [0.35, 0.20, 0.20, 0.15, 0.10]
    )
    device_type = weighted_choice(
        device_types,
        [0.65, 0.25, 0.10]
    )
    browser = weighted_choice(
        browsers,
        [0.45, 0.30, 0.15, 0.10]
    )
    country = weighted_choice(
        countries,
        [0.55, 0.15, 0.12, 0.10, 0.08]
    )

    session_start = random_datetime(start_date, end_date)
    session_duration_seconds = int(np.random.normal(280, 120))
    session_duration_seconds = max(40, session_duration_seconds)

    total_clicks = max(1, int(np.random.poisson(7)))

    sessions.append({
        "session_id": session_id,
        "user_role": user_role,
        "device_type": device_type,
        "browser": browser,
        "country": country,
        "session_start": session_start,
        "session_duration_seconds": session_duration_seconds,
        "total_clicks": total_clicks
    })

sessions_df = pd.DataFrame(sessions)

### Generate user events table

In [6]:
events = []
event_counter = 1

for _, session in sessions_df.iterrows():
    session_id = session["session_id"]
    user_role = session["user_role"]
    product_name = weighted_choice(
        product_names,
        [0.45, 0.15, 0.15, 0.15, 0.10]
    )

    session_start = session["session_start"]
    total_clicks = session["total_clicks"]

    for click_number in range(total_clicks):
        event_timestamp = session_start + timedelta(
            seconds=random.randint(5, session["session_duration_seconds"])
        )

        event_type = weighted_choice(
            ["page_view", "tab_click", "document_click", "export", "verification", "login", "error"],
            [0.12, 0.38, 0.20, 0.10, 0.08, 0.05, 0.07]
        )

        page_section = None
        element_clicked = None
        status = "success"
        error_type = "none"

        if event_type == "page_view":
            page_section = "DPP Landing Page"
            element_clicked = "Open DPP Page"

        elif event_type == "tab_click":
            page_section = weighted_choice(
                tabs,
                [0.25, 0.12, 0.12, 0.22, 0.12, 0.10, 0.07]
            )
            element_clicked = page_section

        elif event_type == "document_click":
            page_section = "Technical Documents"
            element_clicked = weighted_choice(
                documents,
                [0.25, 0.20, 0.25, 0.08, 0.12, 0.10]
            )

        elif event_type == "export":
            page_section = "Export Product Passport"
            element_clicked = weighted_choice(
                exports,
                [0.55, 0.30, 0.15]
            )

        elif event_type == "verification":
            page_section = "Verify Product Passport"
            element_clicked = "Verify with Spherity"

        elif event_type == "login":
            page_section = "Authentication"
            element_clicked = "Log in"

        elif event_type == "error":
            page_section = weighted_choice(
                ["Technical Documents", "Export Product Passport", "Verify Product Passport", "Energy Overview", "Machine Energy"],
                [0.30, 0.25, 0.20, 0.15, 0.10]
            )
            element_clicked = "System Error"

        # Add realistic failure probabilities
        if event_type in ["document_click", "export", "verification", "login", "error"]:
            fail_probability = {
                "document_click": 0.08,
                "export": 0.10,
                "verification": 0.12,
                "login": 0.15,
                "error": 1.00
            }[event_type]

            if random.random() < fail_probability:
                status = "failed"

                if event_type == "document_click":
                    error_type = "broken_link"
                elif event_type == "export":
                    error_type = "failed_export"
                elif event_type == "verification":
                    error_type = "failed_verification"
                elif event_type == "login":
                    error_type = "access_denied"
                else:
                    error_type = weighted_choice(
                        ["api_timeout", "missing_data", "broken_link"],
                        [0.45, 0.35, 0.20]
                    )

        events.append({
            "event_id": f"E{event_counter:05d}",
            "session_id": session_id,
            "user_role": user_role,
            "product_name": product_name,
            "event_timestamp": event_timestamp,
            "event_type": event_type,
            "page_section": page_section,
            "element_clicked": element_clicked,
            "status": status,
            "error_type": error_type
        })

        event_counter += 1

user_events_df = pd.DataFrame(events)


### Generate data quality issues table

In [7]:
quality_issues = []

issue_types = ["missing", "outdated", "invalid_value", "broken_link", "incomplete"]
severity_levels = ["low", "medium", "high"]

fields_by_section = {
    "General Information": ["Product Name", "MPN", "Company Name", "Company Identifier"],
    "Paint Material": ["Hiding Power", "Density", "Spreading Rate", "Hazard Classification"],
    "Packaging": ["Product Carbon Footprint", "Post-Consumer Recycling Content"],
    "Technical Documents": ["Safety Data Sheet", "Sustainability Data Sheet", "Technical Information"],
    "Energy Overview": ["Total Energy Consumption", "Energy Intensity", "Peak Load"],
    "Machine Energy": ["Machine Sensor Reading", "Furnace Energy Usage", "Compressor Energy Usage"],
    "Anomalies": ["Anomaly Score", "Alert Status", "Sensor Error Code"]
}

num_issues = 120

for i in range(1, num_issues + 1):
    section = weighted_choice(
        list(fields_by_section.keys()),
        [0.12, 0.10, 0.12, 0.24, 0.15, 0.15, 0.12]
    )

    field_name = random.choice(fields_by_section[section])
    issue_type = weighted_choice(
        issue_types,
        [0.30, 0.20, 0.15, 0.20, 0.15]
    )

    severity = weighted_choice(
        severity_levels,
        [0.40, 0.40, 0.20]
    )

    detected_date = random_datetime(start_date, end_date).date()

    quality_issues.append({
        "issue_id": f"Q{i:04d}",
        "product_name": weighted_choice(
            product_names,
            [0.45, 0.15, 0.15, 0.15, 0.10]
        ),
        "section": section,
        "field_name": field_name,
        "issue_type": issue_type,
        "severity": severity,
        "detected_date": detected_date
    })

data_quality_df = pd.DataFrame(quality_issues)

### Save CSV files

In [8]:
sessions_df.to_csv("sessions.csv", index=False)
user_events_df.to_csv("user_events.csv", index=False)
data_quality_df.to_csv("data_quality_issues.csv", index=False)

print("Mockup data generated successfully!")
print("Files created:")
print("1. sessions.csv")
print("2. user_events.csv")
print("3. data_quality_issues.csv")

print("\nPreview: user_events.csv")
display(user_events_df.head())

print("\nPreview: sessions.csv")
display(sessions_df.head())

print("\nPreview: data_quality_issues.csv")
display(data_quality_df.head())

Mockup data generated successfully!
Files created:
1. sessions.csv
2. user_events.csv
3. data_quality_issues.csv

Preview: user_events.csv


,event_id,session_id,user_role,product_name,event_timestamp,event_type,page_section,element_clicked,status,error_type
0,E00001,S0001,Manufacturer,Steel Coil A,2026-07-02 05:56:40,page_view,DPP Landing Page,Open DPP Page,success,none
1,E00002,S0001,Manufacturer,Steel Coil A,2026-07-02 05:55:20,tab_click,Technical Documents,Technical Documents,success,none
2,E00003,S0001,Manufacturer,Steel Coil A,2026-07-02 05:56:14,document_click,Technical Documents,Interoperability Test Case Documentation,success,none
3,E00004,S0001,Manufacturer,Steel Coil A,2026-07-02 05:52:45,document_click,Technical Documents,Interoperability Test Case Documentation,success,none
4,E00005,S0002,Manufacturer,The White,2026-07-01 08:42:53,page_view,DPP Landing Page,Open DPP Page,success,none



Preview: sessions.csv


,session_id,user_role,device_type,browser,country,session_start,session_duration_seconds,total_clicks
0,S0001,Manufacturer,Desktop,Chrome,Germany,2026-07-02 05:51:13,339,4
1,S0002,Manufacturer,Mobile,Chrome,Germany,2026-07-01 08:40:44,263,7
2,S0003,Customer,Desktop,Safari,Netherlands,2026-07-07 14:43:32,211,6
3,S0004,Auditor,Desktop,Chrome,Italy,2026-07-02 22:30:14,169,6
4,S0005,Manufacturer,Desktop,Chrome,Egypt,2026-07-05 02:02:24,207,5



Preview: data_quality_issues.csv


,issue_id,product_name,section,field_name,issue_type,severity,detected_date
0,Q0001,Steel Coil A,Paint Material,Spreading Rate,broken_link,high,2026-07-01
1,Q0002,The White,Energy Overview,Peak Load,broken_link,low,2026-07-04
2,Q0003,The White,General Information,Company Identifier,outdated,high,2026-07-06
3,Q0004,The White,Technical Documents,Technical Information,outdated,low,2026-07-07
4,Q0005,The White,General Information,Product Name,incomplete,low,2026-07-05


In [1]:
import os
os.getcwd()

'/Users/mahmoudmansour'